# Notebook 03: Model Training and Evaluation

In this notebook, we train multiple machine learning models and select the best model for movie review sentiment prediction.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report

from utils import evaluate_model, plot_confusion_matrix

In [ ]:
df = pd.read_csv("../data/imdb_cleaned.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=["clean_review", "sentiment"])
df.shape

In [ ]:
X = df["clean_review"]
y = df["sentiment"]

print(X.head())
print(y.head())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Linear SVM": LinearSVC(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [ ]:
results = []
trained_pipelines = {}

for model_name, model in models.items():
    print("Training:", model_name)

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2)
        )),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    result = evaluate_model(model_name, y_test, y_pred)
    results.append(result)

    trained_pipelines[model_name] = pipeline

    print(model_name, "completed")
    print("-" * 50)

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1 Score", ascending=False)

results_df

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="F1 Score", y="Model")
plt.title("Model Comparison by F1 Score")
plt.xlabel("F1 Score")
plt.ylabel("Model")
plt.xlim(0, 1)
plt.show()

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_pipelines[best_model_name]

print("Best Model:", best_model_name)

In [ ]:
y_pred_best = best_model.predict(X_test)

print(classification_report(y_test, y_pred_best))

In [ ]:
plot_confusion_matrix(
    y_test,
    y_pred_best,
    title=f"Confusion Matrix - {best_model_name}"
)

In [ ]:
sample_reviews = [
    "This movie was amazing. The acting was excellent and the story was beautiful.",
    "This was the worst movie I have ever watched. It was boring and too long.",
    "The movie was not good. I expected much better.",
    "The film started slowly, but the ending was emotional and powerful."
]

for review in sample_reviews:
    prediction = best_model.predict([review])[0]

    print("Review:", review)
    print("Prediction:", prediction)
    print("-" * 80)

In [ ]:
wrong_predictions = pd.DataFrame({
    "review": X_test,
    "actual": y_test,
    "predicted": y_pred_best
})

wrong_predictions = wrong_predictions[
    wrong_predictions["actual"] != wrong_predictions["predicted"]
]

wrong_predictions.head(10)

In [ ]:
os.makedirs("../models", exist_ok=True)

joblib.dump(best_model, "../models/movie_sentiment_pipeline.pkl")

print("Best model saved successfully!")
print("Saved model:", best_model_name)

## Model Training Summary

- TF-IDF was used to convert cleaned text into numerical features.
- Four machine learning models were trained:
  1. Multinomial Naive Bayes
  2. Logistic Regression
  3. Linear SVM
  4. Random Forest

- The models were compared using Accuracy, Precision, Recall, and F1 Score.
- The best model was saved as `movie_sentiment_pipeline.pkl`.
- This saved model can be used for real-time prediction using a Streamlit app.